# Preprocess Immune Human Dataset for Geneformer

Map gene symbols to Ensembl IDs, ensure `n_counts` is present, write the
prepared h5ad to a tokenizer-input directory, then run
`TranscriptomeTokenizer` to produce a HuggingFace `Dataset` on disk.

The resulting `.dataset` directory is consumed by `geneformer_encoding.py`.

In [1]:
from pathlib import Path

import scanpy as sc
from datasets import load_from_disk

from src.geneformer import (
    prepare_adata_for_geneformer,
    tokenize_adata,
)

/home/chasty2/Documents/scFM_benchmarking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
DATA_DIR = Path("../data")
INPUT_H5AD = DATA_DIR / "Immune_ALL_human.h5ad"
PREP_DIR = DATA_DIR / "geneformer_prep"
TOKENIZED_DIR = DATA_DIR / "geneformer_tokenized"
OUTPUT_PREFIX = "immune_human"
CELLTYPE_OBS_KEY = "final_annotation"

PREP_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZED_DIR.mkdir(parents=True, exist_ok=True)

## Load AnnData

In [3]:
adata = sc.read(INPUT_H5AD)
print(f"Loaded {adata.shape[0]} cells, {adata.shape[1]} genes")

Loaded 33506 cells, 12303 genes


## Add `ensembl_id` and `n_counts`

In [4]:
adata = prepare_adata_for_geneformer(adata)

n_mapped = (adata.var["ensembl_id"] != "unknown").sum()
print(f"Mapped {n_mapped}/{adata.n_vars} gene symbols to Ensembl IDs")
print(f"obs has n_counts: {'n_counts' in adata.obs.columns}")

Mapped 12014/12303 gene symbols to Ensembl IDs
obs has n_counts: True


## Write prepared h5ad to the tokenizer-input directory

In [5]:
prep_path = PREP_DIR / INPUT_H5AD.name
adata.write(prep_path)
print(f"Wrote {prep_path}")

Wrote ../data/geneformer_prep/Immune_ALL_human.h5ad


## Tokenize

In [6]:
dataset_path = tokenize_adata(
    input_dir=PREP_DIR,
    output_dir=TOKENIZED_DIR,
    output_prefix=OUTPUT_PREFIX,
    custom_attr_name_dict={CELLTYPE_OBS_KEY: "cell_type"},
    nproc=4,
)
print(f"Tokenized dataset → {dataset_path}")

Tokenizing ../data/geneformer_prep/Immune_ALL_human.h5ad


/home/chasty2/Documents/scFM_benchmarking/.venv/lib/python3.12/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/home/chasty2/Documents/scFM_benchmarking/.venv/lib/python3.12/site-packages/geneformer/tokenizer.py:547: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


../data/geneformer_prep/Immune_ALL_human.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
Tokenized dataset → ../data/geneformer_tokenized/immune_human.dataset


## Summarise tokenized dataset

In [7]:
import collections

tokenized = load_from_disk(str(dataset_path))
print(f"Total cells: {len(tokenized)}")
print(f"Columns: {tokenized.column_names}")

if "cell_type" in tokenized.column_names:
    counts = collections.Counter(tokenized["cell_type"])
    print("\nCells per cell type:")
    for ct, n in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"  {n:>6}  {ct}")

Total cells: 33506
Columns: ['input_ids', 'cell_type', 'length']

Cells per cell type:
   11011  CD4+ T cells
    6483  CD14+ Monocytes
    2873  CD20+ B cells
    2745  NKT cells
    2294  NK cells
    2183  CD8+ T cells
    1502  Erythrocytes
    1012  Monocyte-derived dendritic cells
     997  CD16+ Monocytes
     473  HSPCs
     463  Erythroid progenitors
     436  Plasmacytoid dendritic cells
     428  Monocyte progenitors
     270  Megakaryocyte progenitors
     207  CD10+ B cells
     129  Plasma cells
